# Custom VPL Pipeline (Embeddings + Training)

This notebook executes the full VPL pipeline using python shell commands. 
Adjust the variables in the first cell, then run the subsequent cells.

In [ ]:
# --- Configuration ---
model_name = "gpt2"  # or your desired model
data_path = "data/custom_vpl_balanced"
output_dir = "results/custom_vae_model"
subset_size = 4000
pool_size = 100
context_length = 8

# Input Paths (Relative to notebook root)
train_a_path = "data/grouped_data/Persona_A/train.jsonl"
train_b_path = "data/grouped_data/Persona_B/train.jsonl"
test_a_path = "data/grouped_data/Persona_A/test.jsonl"
test_b_path = "data/grouped_data/Persona_B/test.jsonl"

# Training Params (Optimized for L4 GPU)
batch_size = 8
grad_accum = 2
epochs = 3
lr = 3e-6

## 1. Embedding Generation (Train Split)

In [ ]:
!python -m hidden_context.data_utils.custom_vpl_dataset_gen \
    --data_path "." \
    --persona_a_path {train_a_path} \
    --persona_b_path {train_b_path} \
    --output_dir {data_path} \
    --model_type {model_name} \
    --with_embeddings True \
    --subset_size {subset_size} \
    --pool_size {pool_size} \
    --context_length {context_length} \
    --num_duplicates 2 \
    --data_split "train"

## 2. Embedding Generation (Test Split)

In [ ]:
!python -m hidden_context.data_utils.custom_vpl_dataset_gen \
    --data_path "." \
    --persona_a_path {test_a_path} \
    --persona_b_path {test_b_path} \
    --output_dir {data_path} \
    --model_type {model_name} \
    --with_embeddings True \
    --subset_size {subset_size} \
    --pool_size {pool_size} \
    --context_length {context_length} \
    --num_duplicates 1 \
    --data_split "test"

## 3. VAE Training
Optimized for L4 GPU (Batch Size 8, Gradient Accumulation 2).

In [ ]:
!python -m hidden_context.train_llm_vae_preference_model \
    --model_name {model_name} \
    --data_path {data_path} \
    --data_subset "all" \
    --other_subsets "custom_personas" \
    --output_dir {output_dir} \
    --per_device_train_batch_size {batch_size} \
    --per_device_eval_batch_size {batch_size} \
    --gradient_accumulation_steps {grad_accum} \
    --train_dataset_size {subset_size} \
    --eval_dataset_size 1000 \
    --learning_rate {lr} \
    --weight_decay 0.001 \
    --num_train_epochs {epochs} \
    --logging_steps 10 \
    --save_strategy "epoch" \
    --evaluation_strategy "epoch" \
    --fixed_llm_embeddings True \
    --use_last_token_embedding True \
    --remove_unused_columns False